## Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


## Preprocess

In [ ]:
def load_demographics(csv_path):
    """Load demographics data with severity ratings"""
    # df = pd.read_excel(csv_path)
    df = pd.read_excel(csv_path, engine="xlrd")
    # Convert HoehnYahr to categorical values (0=healthy, 1=stage 2, 2=stage 2.5, 3=stage 3)
    severity_map = {
        0: 0,  # Healthy controls
        2.0: 1,    # Stage 2
        2.5: 2,    # Stage 2.5
        3.0: 3     # Stage 3
    }

    # Map severity from HoehnYahr column
    df['severity_class'] = df['HoehnYahr'].map(severity_map)
    return df

def extract_file_identifiers(filename):
    """Extract Study and Subject Number from filename like 'GaCo02_01.txt'"""
    # For filenames like GaCo02_01.txt
    parts = filename.split('_')[0]
    study = parts[:2]  # Extract study code (Ga)

    # Check if file is from control group or PD group
    if 'Co' in parts:
        group = 'CO'  # Control group
    else:
        group = 'PD'  # PD group

    # Extract subject number
    subjnum = int(parts[4:6] if len(parts) >= 6 else parts[2:4])

    return study, group, subjnum


def preprocess_file(file_path, sequence_length=1000):
    # Load tab-separated file with no header
    data = pd.read_csv(file_path, sep='\t', header=None)

    # Sanity check: need at least 18 columns (0-based index 17 present)
    if data.shape[1] < 18:
        raise ValueError(f"{file_path} has only {data.shape[1]} columns; expected ≥ 18.")

    # Select only Column 18 (Total force under the left foot) -> keep 2D shape (T, 1)
    features = data.iloc[:, [18]].to_numpy()   # shape: (T, 1).  Use [[17]] not [17] to keep 2D

    T = features.shape[0]
    segments = []

    if T >= sequence_length:
        # 50% overlap
        step_size = sequence_length // 2
        for i in range(0, T - sequence_length + 1, step_size):
            seg = features[i:i+sequence_length, :]   # (L, 1)
            seg = seg.T                               # (1, L) -> channels-first
            segments.append(seg)
    else:
        # Pad if too short
        pad_len = sequence_length - T
        padding = np.zeros((pad_len, 1), dtype=features.dtype)
        seg = np.vstack([features, padding]).T       # (1, L)
        segments.append(seg)

    return segments


def load_all_data(data_dir, demographics_df, sequence_length=1000):
    """Load and preprocess all gait files with labels"""
    X_data = []
    y_data = []
    file_count = 0

    for file in os.listdir(data_dir):
        if not file.endswith('.txt'):
            continue

        # Extract identifiers from filename
        study, group, subjnum = extract_file_identifiers(file)

        # Find matching row in demographics
        matching_row = demographics_df[(demographics_df['Study'] == study) &
                                        (demographics_df['Group'] == group) &
                                        (demographics_df['Subjnum'] == subjnum)]

        if matching_row.empty:
            print(f"No demographic data for {file}")
            continue

        # Get severity class
        severity = matching_row['severity_class'].values[0]

        # Process file
        file_path = os.path.join(data_dir, file)
        segments = preprocess_file(file_path, sequence_length)

        if segments is not None:
            for segment in segments:
                X_data.append(segment)
                y_data.append(severity)

            file_count += 1
            if file_count % 50 == 0:
                print(f"Processed {file_count} files")

    return np.array(X_data), np.array(y_data)


## Data class

In [ ]:
class ParkinsonsGaitDataset(Dataset):
    """Dataset class for Parkinson's gait data from PhysioNet"""

    def __init__(self, X_data, y_data, transform=None):
        """
        Args:
            X_data: Preprocessed feature data
            y_data: Severity labels
            transform: Optional transform to apply to the data
        """
        self.X_data = X_data
        self.y_data = y_data
        self.transform = transform

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        # Get data and label
        data = self.X_data[idx]
        label = self.y_data[idx]

        # Convert to tensor
        data = torch.FloatTensor(data)
        label = torch.LongTensor([label])  # Use LongTensor for classification

        if self.transform:
            data = self.transform(data)

        return data, label

## Train setup

In [ ]:
def train_model(model, train_loader, val_loader,
                num_epochs=20, learning_rate=0.001,
                patience=3, min_delta=0.0):
    """Train the CNN model with inline early stopping (on val loss) and checkpoint on best val F1."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

    train_losses = []
    val_losses = []
    train_f1_scores = []
    val_f1_scores = []

    # --- Early stopping / best checkpoint state ---
    best_val_loss = None
    bad_epochs = 0
    best_val_f1 = -float("inf")
    best_state = None
    best_epoch = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        # Training phase
        model.train()
        train_loss = 0.0
        train_predictions = []
        train_labels = []

        for batch_data, batch_labels in train_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_labels = batch_labels.squeeze()

            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Collect predictions and labels for F1 calculation
            _, predicted = torch.max(outputs.data, 1)
            train_predictions.extend(predicted.detach().cpu().numpy())
            train_labels.extend(batch_labels.detach().cpu().numpy())

        # Calculate training F1-score (weighted)
        train_f1 = f1_score(train_labels, train_predictions, average='weighted', zero_division=0)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_labels = []

        with torch.no_grad():
            for batch_data, batch_labels in val_loader:
                batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
                batch_labels = batch_labels.squeeze()

                outputs = model(batch_data)
                loss = criterion(outputs, batch_labels)
                val_loss += loss.item()

                # Collect predictions and labels for F1 calculation
                _, predicted = torch.max(outputs.data, 1)
                val_predictions.extend(predicted.detach().cpu().numpy())
                val_labels.extend(batch_labels.detach().cpu().numpy())

        # Calculate validation F1-score (weighted)
        val_f1 = f1_score(val_labels, val_predictions, average='weighted', zero_division=0)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_f1_scores.append(train_f1)
        val_f1_scores.append(val_f1)

        scheduler.step(avg_val_loss)

        print(f'Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
        print(f'Train F1 (weighted): {train_f1:.4f}, Val F1 (weighted): {val_f1:.4f}')

        # ----- Checkpoint on best Val F1 (maximize) -----
        if val_f1 > best_val_f1 + 1e-6:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            print(f"✅ New best Val F1={best_val_f1:.4f} at epoch {best_epoch}")

        # ----- Early stopping on Val Loss (minimize) -----
        if best_val_loss is None:
            best_val_loss = avg_val_loss
            bad_epochs = 0
        else:
            improved = (avg_val_loss < best_val_loss - min_delta)
            if improved:
                best_val_loss = avg_val_loss
                bad_epochs = 0
            else:
                bad_epochs += 1
                print(f"⚠️  No val loss improvement for {bad_epochs}/{patience} epoch(s).")

        if bad_epochs >= patience:
            print(f"⛔ Early stopping triggered (no val loss improvement ≥ {min_delta} for {patience} epochs).")
            break

    # Restore best-F1 weights for downstream use
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"🔄 Restored best model (epoch {best_epoch}, Val F1={best_val_f1:.4f})")

    return train_losses, val_losses, train_f1_scores, val_f1_scores


In [ ]:
def evaluate_model(model, test_loader, target_avg="weighted"):
    """Evaluate the trained model on a PyTorch DataLoader."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    predictions = []
    actual_values = []

    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_labels = batch_labels.squeeze()  # (B,)

            outputs = model(batch_data)
            _, predicted = torch.max(outputs.data, 1)

            predictions.extend(predicted.cpu().numpy())
            actual_values.extend(batch_labels.cpu().numpy())

    predictions = np.array(predictions)
    actual_values = np.array(actual_values)

    accuracy = accuracy_score(actual_values, predictions)
    f1w = f1_score(actual_values, predictions, average=target_avg)

    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Test F1 ({target_avg}): {f1w:.4f}\n')
    print("Classification Report:")
    print(classification_report(actual_values, predictions,
                               target_names=['Healthy', 'Stage 2', 'Stage 2.5', 'Stage 3']))
    print("\nConfusion Matrix:")
    print(confusion_matrix(actual_values, predictions))

    return predictions, actual_values, f1w


## Model

In [ ]:
class ParkinsonsGaitCNN(nn.Module):
    """
    Same architecture, but now configurable for grid search:
      - num_conv_layers ∈ {2,3,4,5}
      - num_fc_layers   ∈ {1,2,3,4}
      - activation_name ∈ {'relu','leaky_relu','elu'}
      - dropout_p       ∈ {0.3,0.5,0.7}
    """

    def __init__(
        self,
        input_channels: int = 1,
        sequence_length: int = 1000,
        num_conv_layers: int = 4,
        num_fc_layers: int = 2,
        activation_name: str = "relu",
        dropout_p: float = 0.5,
        num_classes: int = 4
    ):
        super().__init__()
        assert num_conv_layers in (2,3,4,5)
        assert num_fc_layers in (1,2,3,4)

        act_map = {
            "relu": nn.ReLU(inplace=True),
            "leaky_relu": nn.LeakyReLU(0.01, inplace=True),
            "elu": nn.ELU(alpha=1.0, inplace=True),
        }
        if activation_name not in act_map:
            raise ValueError(f"Unsupported activation: {activation_name}")
        self.act = act_map[activation_name]

        self.num_conv_layers = num_conv_layers
        self.num_fc_layers = num_fc_layers
        self.dropout = nn.Dropout(dropout_p)

        # ----- Conv blocks (same kernels as your original) -----
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, padding=3)
        self.pool1 = nn.AvgPool1d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool2 = nn.AvgPool1d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.pool3 = nn.AvgPool1d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv1d(256, 512, kernel_size=3, padding=1)
        self.pool4 = nn.AvgPool1d(kernel_size=2, stride=2)

        # NEW: optional 5th conv block (keeps channels at 512)
        self.conv5 = nn.Conv1d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.AvgPool1d(kernel_size=2, stride=2)

        # ----- compute flattened size with selected conv depth -----
        self.flattened_size = self._get_flattened_size(input_channels, sequence_length)

        # ----- FC head: keep your sizes, allow 1/2/3/4 layers -----
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        # NEW: 4th FC for num_fc_layers==4
        self.fc4 = nn.Linear(64, 32)

        if num_fc_layers == 1:
            final_in = 256
        elif num_fc_layers == 2:
            final_in = 128
        elif num_fc_layers == 3:
            final_in = 64
        else:  # num_fc_layers == 4
            final_in = 32

        self.fc_out = nn.Linear(final_in, num_classes)

    def _get_flattened_size(self, input_channels: int, sequence_length: int) -> int:
        x = torch.randn(1, input_channels, sequence_length)
        x = self.pool1(self.act(self.conv1(x)))
        x = self.pool2(self.act(self.conv2(x)))
        if self.num_conv_layers >= 3:
            x = self.pool3(self.act(self.conv3(x)))
        if self.num_conv_layers >= 4:
            x = self.pool4(self.act(self.conv4(x)))
        if self.num_conv_layers >= 5:
            x = self.pool5(self.act(self.conv5(x)))
        return x.numel()

    def forward(self, x):
        x = self.pool1(self.act(self.conv1(x)))
        x = self.pool2(self.act(self.conv2(x)))
        if self.num_conv_layers >= 3:
            x = self.pool3(self.act(self.conv3(x)))
        if self.num_conv_layers >= 4:
            x = self.pool4(self.act(self.conv4(x)))
        if self.num_conv_layers >= 5:
            x = self.pool5(self.act(self.conv5(x)))

        x = x.view(x.size(0), -1)

        # 1/2/3/4 FC layers (each with dropout + activation)
        x = self.dropout(x); x = self.act(self.fc1(x))
        if self.num_fc_layers >= 2:
            x = self.dropout(x); x = self.act(self.fc2(x))
        if self.num_fc_layers >= 3:
            x = self.dropout(x); x = self.act(self.fc3(x))
        if self.num_fc_layers >= 4:
            x = self.dropout(x); x = self.act(self.fc4(x))

        x = self.fc_out(x)
        return x


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV

class TorchGaitClassifier(BaseEstimator, ClassifierMixin):
    """
    Minimal sklearn estimator wrapper around your PyTorch model + train loop.
    Works with GridSearchCV: supports set_params/get_params via BaseEstimator.
    Trains with an internal 80/20 split to drive your existing train_model (needs val_loader).
    """
    def __init__(
        self,
        sequence_length=1000,
        num_epochs=20,
        batch_size=64,
        learning_rate=1e-3,
        num_conv_layers=4,
        num_fc_layers=2,
        activation_name="relu",
        dropout_p=0.5,
        random_state=42,
        early_stop_patience=3,
        early_stop_min_delta=0.0
    ):
        self.sequence_length = sequence_length
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.num_conv_layers = num_conv_layers
        self.num_fc_layers = num_fc_layers
        self.activation_name = activation_name
        self.dropout_p = dropout_p
        self.random_state = random_state
        self.model_ = None
        self.input_channels_ = None,
        self.early_stop_patience = early_stop_patience
        self.early_stop_min_delta = early_stop_min_delta

    def _make_loader(self, X, y, idx, shuffle=False):
        ds = ParkinsonsGaitDataset(X[idx], y[idx])
        return DataLoader(ds, batch_size=self.batch_size, shuffle=shuffle)

    def fit(self, X, y):
        # X expected: (N, C, L), y: (N,)
        rng = np.random.RandomState(self.random_state)
        n = len(y)
        idx = np.arange(n)
        rng.shuffle(idx)
        split = int(0.8 * n) if n >= 5 else max(1, n-1)
        tr_idx, va_idx = idx[:split], idx[split:]

        self.input_channels_ = X.shape[1]
        model = ParkinsonsGaitCNN(
            input_channels=self.input_channels_,
            sequence_length=self.sequence_length,
            num_conv_layers=self.num_conv_layers,
            num_fc_layers=self.num_fc_layers,
            activation_name=self.activation_name,
            dropout_p=self.dropout_p,
            num_classes=4
        )

        train_loader = self._make_loader(X, y, tr_idx, shuffle=True)
        val_loader   = self._make_loader(X, y, va_idx, shuffle=False)

        # Train with existing helper
        train_model(
            model, train_loader, val_loader,
            num_epochs=self.num_epochs,
            learning_rate=self.learning_rate,
            patience=self.early_stop_patience,
            min_delta=self.early_stop_min_delta
        )

        self.model_ = model
        return self

    def predict(self, X):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_.to(device)
        self.model_.eval()
        preds = []
        with torch.no_grad():
            # make a simple loader for prediction
            ds = ParkinsonsGaitDataset(X, np.zeros(len(X)))  # labels not used
            loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False)
            for xb, _ in loader:
                xb = xb.to(device)
                logits = self.model_(xb)
                pred = torch.argmax(logits, dim=1).cpu().numpy()
                preds.append(pred)
        return np.concatenate(preds)

    def score(self, X, y):
        # accuracy to match your RF example
        yhat = self.predict(X)
        return accuracy_score(y, yhat)


## Data Config & Preprocess

In [ ]:
"""Main function to run the training pipeline"""
# Configuration
DATA_DIR = "dataset/train"
DEMOGRAPHICS_PATH = "dataset/demographics.xls"
SEQUENCE_LENGTH = 1000
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
EARLY_STOP_PATIENCE = 3

In [ ]:
# Load demographics data
print("Loading demographics data...")
demographics_df = load_demographics(DEMOGRAPHICS_PATH)
print(f"Demographics loaded: {len(demographics_df)} records")

Loading demographics data...
Demographics loaded: 166 records


In [ ]:
# Load and preprocess all data
print("Loading and preprocessing gait data...")
X_data, y_data = load_all_data(DATA_DIR, demographics_df, SEQUENCE_LENGTH)
print(f"Data loaded: {X_data.shape[0]} samples, {X_data.shape[1]} features, {X_data.shape[2]} sequence length")

Loading and preprocessing gait data...
Processed 50 files
Processed 100 files
Processed 150 files
Processed 200 files
Processed 250 files
Processed 300 files
Data loaded: 6229 samples, 1 features, 1000 sequence length


In [ ]:
# Print class distribution
unique, counts = np.unique(y_data, return_counts=True)
print("Class distribution:")
class_names = ['Healthy', 'Stage 2', 'Stage 2.5', 'Stage 3']
for i, count in enumerate(counts):
    print(f"  {class_names[i]}: {count} samples")

Class distribution:
  Healthy: 1909 samples
  Stage 2: 2269 samples
  Stage 2.5: 1518 samples
  Stage 3: 533 samples


## Dataloader

In [ ]:
# Create dataset
dataset = ParkinsonsGaitDataset(X_data, y_data)

# Split dataset
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# We'll build loaders for the final test later; for CV, we need raw arrays of train+val
train_indices = np.array(train_dataset.indices)
val_indices = np.array(val_dataset.indices)
trainval_idx = np.concatenate([train_indices, val_indices])

X_trainval = X_data[trainval_idx]
y_trainval = y_data[trainval_idx]

# Final test DataLoader (kept fully held-out)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


Dataset sizes - Train: 4360, Val: 934, Test: 935


## Train (New)

In [ ]:
import copy

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score
from numpy import mean, std

param_grid = {
    "num_conv_layers": [3, 4, 5],
    "num_fc_layers":   [2, 3, 4],
    "activation_name": ["relu", "leaky_relu", "elu"],
    "dropout_p":       [0.3, 0.5, 0.7],
    "learning_rate":   [1e-3, 3e-4, 1e-4],
}

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_results = []
outer_best = {"score": -1.0, "params": None}

print("\n===== Nested CV on train+val (scoring=f1_weighted) =====")
fold_no = 0
for train_ix, test_ix in cv_outer.split(X_trainval, y_trainval):
    fold_no += 1
    print(f"\n[Outer Fold {fold_no}]")
    X_tr, X_te = X_trainval[train_ix], X_trainval[test_ix]
    y_tr, y_te = y_trainval[train_ix], y_trainval[test_ix]

    cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    base_est = TorchGaitClassifier(
        sequence_length=SEQUENCE_LENGTH,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,  # overridden by grid
        num_conv_layers=4,
        num_fc_layers=2,
        activation_name="relu",
        dropout_p=0.5,
        random_state=42,
        early_stop_patience=EARLY_STOP_PATIENCE
    )

    search = GridSearchCV(
        estimator=base_est,
        param_grid=param_grid,
        scoring='f1_weighted',
        cv=cv_inner,
        refit=True,
        n_jobs=1,  # GPU contention safety
        verbose=3
    )

    result = search.fit(X_tr, y_tr)
    best_model = result.best_estimator_
    yhat = best_model.predict(X_te)
    f1w = f1_score(y_te, yhat, average='weighted')
    outer_results.append(f1w)

    print('>F1_weighted=%.3f, est(inner)=%.3f, cfg=%s' %
          (f1w, result.best_score_, result.best_params_))

    if f1w > outer_best["score"]:
        outer_best["score"] = f1w
        outer_best["params"] = result.best_params_

print('Outer F1_weighted: %.3f (%.3f)' % (mean(outer_results), std(outer_results)))
print(f"Best outer-fold params: {outer_best['params']}")


Streaming output truncated to the last 5000 lines.
🔄 Restored best model (epoch 3, Val F1=0.4538)
[CV 3/3] END activation_name=elu, dropout_p=0.7, learning_rate=0.001, num_conv_layers=5, num_fc_layers=2;, score=0.430 total time=   2.8s

Epoch [1/20]
Train Loss: 1.5306, Val Loss: 1.3053
Train F1 (weighted): 0.3126, Val F1 (weighted): 0.2620
✅ New best Val F1=0.2620 at epoch 1

Epoch [2/20]
Train Loss: 1.3922, Val Loss: 1.3020
Train F1 (weighted): 0.2956, Val F1 (weighted): 0.2838
✅ New best Val F1=0.2838 at epoch 2

Epoch [3/20]
Train Loss: 1.3190, Val Loss: 1.2628
Train F1 (weighted): 0.3403, Val F1 (weighted): 0.2155

Epoch [4/20]
Train Loss: 1.3244, Val Loss: 1.2269
Train F1 (weighted): 0.3303, Val F1 (weighted): 0.3281
✅ New best Val F1=0.3281 at epoch 4

Epoch [5/20]
Train Loss: 1.2919, Val Loss: 1.1793
Train F1 (weighted): 0.3538, Val F1 (weighted): 0.4000
✅ New best Val F1=0.4000 at epoch 5

Epoch [6/20]
Train Loss: 1.2413, Val Loss: 1.1570
Train F1 (weighted): 0.3867, Val F1 (we

In [ ]:
# Refit final model on all train+val using the best params found in nested CV
final_est = TorchGaitClassifier(
    sequence_length=SEQUENCE_LENGTH,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=outer_best['params']['learning_rate'],
    num_conv_layers=outer_best['params']['num_conv_layers'],
    num_fc_layers=outer_best['params']['num_fc_layers'],
    activation_name=outer_best['params']['activation_name'],
    dropout_p=outer_best['params']['dropout_p'],
    random_state=42,
    early_stop_patience=EARLY_STOP_PATIENCE
).fit(X_trainval, y_trainval)



Epoch [1/20]
Train Loss: 1.3103, Val Loss: 1.2229
Train F1 (weighted): 0.2879, Val F1 (weighted): 0.2325
✅ New best Val F1=0.2325 at epoch 1

Epoch [2/20]
Train Loss: 1.2640, Val Loss: 1.1913
Train F1 (weighted): 0.2657, Val F1 (weighted): 0.2711
✅ New best Val F1=0.2711 at epoch 2

Epoch [3/20]
Train Loss: 1.2476, Val Loss: 1.2219
Train F1 (weighted): 0.2716, Val F1 (weighted): 0.2840
✅ New best Val F1=0.2840 at epoch 3
⚠️  No val loss improvement for 1/3 epoch(s).

Epoch [4/20]
Train Loss: 1.2040, Val Loss: 1.1345
Train F1 (weighted): 0.3303, Val F1 (weighted): 0.3383
✅ New best Val F1=0.3383 at epoch 4

Epoch [5/20]
Train Loss: 1.1163, Val Loss: 0.9468
Train F1 (weighted): 0.4497, Val F1 (weighted): 0.5437
✅ New best Val F1=0.5437 at epoch 5

Epoch [6/20]
Train Loss: 0.9750, Val Loss: 0.8731
Train F1 (weighted): 0.5475, Val F1 (weighted): 0.5935
✅ New best Val F1=0.5935 at epoch 6

Epoch [7/20]
Train Loss: 0.9444, Val Loss: 0.7955
Train F1 (weighted): 0.5708, Val F1 (weighted): 0.6

In [ ]:
# Save trained weights from the refit estimator
torch.save(final_est.model_.state_dict(), "saved_weight_251018.pth")
print("Saved final model weights to saved_weight_251018.pth")

Saved final model weights to saved_weight_251018.pth


## Eval (New)

In [ ]:
final_est.model_.load_state_dict(torch.load("saved_weight_251018.pth"))

<All keys matched successfully>

In [ ]:
print("Evaluating model on held-out test set...")
# Option 1 (direct, simplest): use the refit estimator's trained PyTorch model
predictions, actual_values, test_f1w = evaluate_model(final_est.model_, test_loader, target_avg="weighted")

Evaluating model on held-out test set...
Test Accuracy: 0.8877
Test F1 (weighted): 0.8879

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.92      0.90      0.91       278
     Stage 2       0.85      0.89      0.87       346
   Stage 2.5       0.90      0.88      0.89       228
     Stage 3       0.92      0.84      0.88        83

    accuracy                           0.89       935
   macro avg       0.90      0.88      0.89       935
weighted avg       0.89      0.89      0.89       935


Confusion Matrix:
[[251  23   3   1]
 [ 19 309  15   3]
 [  2  24 200   2]
 [  1   7   5  70]]


In [ ]:
print("Evaluating model on held-out test set...")
# Option 1 (direct, simplest): use the refit estimator's trained PyTorch model
predictions, actual_values, test_f1w = evaluate_model(final_est.model_, test_loader, target_avg="weighted")

Evaluating model on held-out test set...
Test Accuracy: 0.6428
Test F1 (weighted): 0.6350

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.68      0.85      0.76       300
     Stage 2       0.60      0.59      0.59       334
   Stage 2.5       0.69      0.52      0.59       223
     Stage 3       0.52      0.42      0.47        78

    accuracy                           0.64       935
   macro avg       0.62      0.60      0.60       935
weighted avg       0.64      0.64      0.64       935


Confusion Matrix:
[[256  33   6   5]
 [ 84 197  37  16]
 [ 21  78 115   9]
 [ 15  21   9  33]]
